Ingestão de Dados - Yellow Taxis NYC (Jan-Mai 2023)
Pipeline de ingestão seguindo a Arquitetura Medallion:

Landing Zone (Unity Catalog Volume): Arquivos Parquet carregados no Volume do Unity Catalog
Bronze: Dados brutos ingeridos, sem transformação, com metadados de ingestão
Silver: Dados limpos e tipados, com colunas selecionadas, prontos para consumo
1. Configuração e Parâmetros

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
from functools import reduce
from pyspark.sql import DataFrame

Parâmetros do Pipeline
Ajuste o nome do catálogo conforme seu ambiente.

Landing Zone: Volume do Unity Catalog onde os arquivos Parquet foram carregados.

Para carregar os arquivos, use a interface do Databricks:

Vá em Catalog → selecione seu catálogo → schema landing
Crie um Volume chamado yellow_taxi
Faça upload dos arquivos .parquet para esse Volume

In [ ]:
# === AJUSTE ESTES PARÂMETROS CONFORME SEU AMBIENTE ===
CATALOG = "main"  # Nome do catálogo no Unity Catalog

# Caminhos derivados
LANDING_VOLUME = f"/Volumes/{CATALOG}/landing/yellow_taxi"

BRONZE_TABLE = f"{CATALOG}.bronze.yellow_taxi_trips"
SILVER_TABLE = f"{CATALOG}.silver.yellow_taxi_trips"

MESES = ["2023-01", "2023-02", "2023-03", "2023-04", "2023-05"]

REQUIRED_COLUMNS = [
    "VendorID",
    "passenger_count",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]


2. Criação dos Schemas no Unity Catalog

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.landing")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

3. Camada Bronze
Leitura dos dados brutos do Volume (Landing Zone) e escrita na camada Bronze.

Mantém todas as colunas originais
Adiciona metadados de ingestão (_data_ingestao, _arquivo_origem, _ano_mes_referencia)
Salva como tabela gerenciada Delta no Unity Catalog, particionada por _ano_mes_referencia

In [ ]:
def ingest_to_bronze(landing_volume, bronze_table, meses):
    """
    Lê os arquivos Parquet do Volume do Unity Catalog,
    adiciona metadados de ingestão e escreve na camada Bronze.
    """
    dataframes = []

    for mes in meses:
        file_path = f"{landing_volume}/yellow_tripdata_{mes}.parquet"
        print(f"Lendo da Landing Zone (Volume): {file_path}")

        df = spark.read.parquet(file_path)

        df_with_meta = (
            df.withColumn("_data_ingestao", F.current_timestamp())
            .withColumn("_arquivo_origem", F.lit(f"yellow_tripdata_{mes}.parquet"))
            .withColumn("_ano_mes_referencia", F.lit(mes))
        )
        dataframes.append(df_with_meta)

    df_bronze = reduce(DataFrame.unionByName, dataframes)

    print(f"Total de registros para Bronze: {df_bronze.count()}")
    print("Schema Bronze:")
    df_bronze.printSchema()

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("_ano_mes_referencia")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table)
    )

    print("Dados escritos na camada Bronze com sucesso!")
    return df_bronze  

df_bronze = ingest_to_bronze(LANDING_VOLUME, BRONZE_TABLE, MESES)

Verificação Bronze

In [ ]:
display(spark.sql(f"""
    SELECT _ano_mes_referencia, COUNT(*) AS total_registros
    FROM {BRONZE_TABLE}
    GROUP BY _ano_mes_referencia
    ORDER BY _ano_mes_referencia
"""))

4. Camada Silver
Leitura da camada Bronze, seleção das colunas obrigatórias, normalização de tipos, limpeza de dados e escrita na camada Silver.

Transformações aplicadas:

Seleção das 5 colunas obrigatórias
Cast explícito dos tipos (INT, DOUBLE, TIMESTAMP)
Remoção de registros com datas nulas
Remoção de registros com total_amount nulo ou negativo
Remoção de registros com passenger_count negativo
Adição de coluna ano_mes derivada de tpep_pickup_datetime

In [ ]:
def transform_bronze_to_silver(bronze_table, silver_table):
    """
    Lê dados da camada Bronze, aplica transformações e escreve na camada Silver.
    """
    df_bronze = spark.table(bronze_table)
    print(f"Registros lidos da Bronze: {df_bronze.count()}")

    type_map = {
        "VendorID": IntegerType(),
        "passenger_count": DoubleType(),
        "total_amount": DoubleType(),
        "tpep_pickup_datetime": TimestampType(),
        "tpep_dropoff_datetime": TimestampType(),
    }

    df_selected = df_bronze.select(
        *[F.col(c).cast(type_map[c]).alias(c) for c in REQUIRED_COLUMNS]
    )

    df_clean = df_selected.filter(
        F.col("tpep_pickup_datetime").isNotNull()
        & F.col("tpep_dropoff_datetime").isNotNull()
        & F.col("total_amount").isNotNull()
        & (F.col("total_amount") >= 0)
        & (F.col("passenger_count") >= 0)
    )

    df_silver = df_clean.withColumn(
        "ano_mes", F.date_format(F.col("tpep_pickup_datetime"), "yyyy-MM")
    )

    registros_removidos = df_bronze.count() - df_silver.count()
    print(f"Registros removidos na limpeza: {registros_removidos}")

    print("Schema Silver:")
    df_silver.printSchema()

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("ano_mes")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    print("Dados escritos na camada Silver com sucesso!")
    return df_silver
    

df_silver = transform_bronze_to_silver(BRONZE_TABLE, SILVER_TABLE)

Verificação Silver

In [ ]:
display(spark.sql(f"""
    SELECT ano_mes, COUNT(*) AS total_registros
    FROM {SILVER_TABLE}
    GROUP BY ano_mes
    ORDER BY ano_mes
"""))

Amostra dos Dados Silver

In [ ]:
display(spark.sql(f"SELECT * FROM {SILVER_TABLE} LIMIT 10"))

5. Resumo da Ingestão

In [ ]:
print("=== RESUMO DA INGESTÃO ===")
bronze_count = spark.table(BRONZE_TABLE).count()
silver_count = spark.table(SILVER_TABLE).count()
print(f"Registros na Bronze: {bronze_count}")
print(f"Registros na Silver: {silver_count}")
print(f"Registros removidos na limpeza: {bronze_count - silver_count}")
print("\nTabelas disponíveis para consulta SQL:")
print(f"  - {BRONZE_TABLE}")
print(f"  - {SILVER_TABLE}")
  